# EDA — Brazilian E-Commerce (Olist)

Análise Exploratória de Dados seguindo a **Fase 2 do CRISP-DM**.  
Dataset: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

**Variável-alvo escolhida:** `delivery_late` — indica se o pedido foi entregue após a data estimada (1 = atrasado, 0 = no prazo).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Carregamento dos Dados

O dataset é composto por 9 arquivos CSV com informações sobre pedidos, clientes, produtos, pagamentos e avaliações.

In [ ]:
DATA_PATH = "data/"

orders     = pd.read_csv(DATA_PATH + "olist_orders_dataset.csv")
customers  = pd.read_csv(DATA_PATH + "olist_customers_dataset.csv")
items      = pd.read_csv(DATA_PATH + "olist_order_items_dataset.csv")
payments   = pd.read_csv(DATA_PATH + "olist_order_payments_dataset.csv")
reviews    = pd.read_csv(DATA_PATH + "olist_order_reviews_dataset.csv")
products   = pd.read_csv(DATA_PATH + "olist_products_dataset.csv")
sellers    = pd.read_csv(DATA_PATH + "olist_sellers_dataset.csv")
categories = pd.read_csv(DATA_PATH + "product_category_name_translation.csv")

In [ ]:
datasets = {
    "orders": orders, "customers": customers, "items": items,
    "payments": payments, "reviews": reviews,
    "products": products, "sellers": sellers, "categories": categories
}

for name, df_aux in datasets.items():
    print(f"{name:12s} → {df_aux.shape[0]:7,} linhas  ×  {df_aux.shape[1]:2} colunas")

## 2. Visão Geral das Tabelas Principais

In [ ]:
orders.head()

In [ ]:
items.head()

In [ ]:
payments.head()

In [ ]:
reviews.head()

## 3. Construção do Dataset Principal

Unimos as principais tabelas em um único DataFrame, onde **cada linha representa um pedido**.

- `items` → agregado por pedido (total de preço, frete e número de itens)
- `payments` → agregado por pedido (valor total e tipo de pagamento mais usado)
- `reviews` → agregado por pedido (nota média)
- `customers` → adicionado o estado do cliente

In [ ]:
items_agg = items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum")
).reset_index()

payments_agg = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_type=("payment_type", lambda x: x.mode()[0]),
    payment_installments=("payment_installments", "max")
).reset_index()

reviews_agg = reviews.groupby("order_id").agg(
    review_score=("review_score", "mean")
).reset_index()

df = (
    orders
    .merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
)

print("Shape final:", df.shape)

In [ ]:
# Converte timestamps
date_cols = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

# Define variável-alvo: entregue após a data estimada?
df["delivery_late"] = (
    (df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]) &
    df["order_delivered_customer_date"].notna()
).astype(int)

print("Distribuição da variável-alvo:")
print(df["delivery_late"].value_counts().rename({0: "No prazo", 1: "Atrasado"}))

In [ ]:
df.info()

## 4. Train/Test Split — A Regra de Ouro

Antes de qualquer análise, separamos os dados em treino e teste para **evitar vazamento de dados** (*data leakage*).  
Toda a EDA a seguir será feita **exclusivamente com `X_train` e `y_train`**.

In [ ]:
drop_cols = [
    "order_id", "customer_id",
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_late"
]

X = df.drop(columns=drop_cols)
y = df["delivery_late"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\nDistribuição do target em y_train:")
print(y_train.value_counts(normalize=True).round(3).rename({0: "No prazo", 1: "Atrasado"}))

In [ ]:
# DataFrame de treino completo (features + target) para uso nas análises
train_df = X_train.copy()
train_df["delivery_late"] = y_train.values

## 5. Análise de Valores Ausentes

Identificamos e quantificamos os NaNs em `X_train`. Lembrando: qualquer estratégia de imputação deve ser aprendida **apenas no treino** e depois aplicada ao teste.

In [ ]:
nan_counts = X_train.isnull().sum()
nan_pct    = (nan_counts / len(X_train) * 100).round(2)

nan_df = pd.DataFrame({"qtd_nan": nan_counts, "pct_nan (%)": nan_pct})
nan_df[nan_df["qtd_nan"] > 0].sort_values("pct_nan (%)", ascending=False)

In [ ]:
plt.figure(figsize=(14, 4))
sns.heatmap(X_train.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Padrão de Valores Ausentes — X_train")
plt.tight_layout()
plt.show()

## 6. Variáveis Numéricas

Estatísticas descritivas e distribuições das principais variáveis numéricas.

In [ ]:
num_cols = ["total_price", "total_freight", "payment_value", "payment_installments", "n_items", "review_score"]
X_train[num_cols].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    X_train[col].dropna().hist(ax=axes[i], bins=40, color="steelblue", edgecolor="white")
    axes[i].set_title(col)

plt.suptitle("Distribuição das Variáveis Numéricas (X_train)", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Variáveis Categóricas

Distribuição das principais variáveis categóricas do conjunto de treino.

In [ ]:
status_counts = X_train["order_status"].value_counts()

plt.figure(figsize=(10, 5))
sns.barplot(x=status_counts.index, y=status_counts.values, palette="viridis")
plt.title("Distribuição de Status dos Pedidos")
plt.xlabel("Status")
plt.ylabel("Contagem")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(
    data=X_train,
    x="payment_type",
    order=X_train["payment_type"].value_counts().index,
    palette="viridis"
)
plt.title("Distribuição de Tipo de Pagamento")
plt.xlabel("Tipo de Pagamento")
plt.ylabel("Contagem")
plt.tight_layout()
plt.show()

In [ ]:
top_states = X_train["customer_state"].value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_states.index, y=top_states.values, palette="viridis")
plt.title("Top 10 Estados por Número de Pedidos")
plt.xlabel("Estado")
plt.ylabel("Contagem")
plt.tight_layout()
plt.show()

In [ ]:
# Usamos o df original para cruzar items → products → categories
# Filtramos apenas os pedidos que estão no conjunto de treino
train_order_ids = df.loc[X_train.index, "order_id"]

train_items_cat = (
    items
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(categories, on="product_category_name", how="left")
)
train_items_cat = train_items_cat[train_items_cat["order_id"].isin(train_order_ids)]

top_cats = train_items_cat["product_category_name_english"].dropna().value_counts().head(10)

plt.figure(figsize=(12, 5))
sns.barplot(x=top_cats.values, y=top_cats.index, palette="viridis")
plt.title("Top 10 Categorias de Produtos (por nº de itens vendidos)")
plt.xlabel("Contagem")
plt.ylabel("Categoria")
plt.tight_layout()
plt.show()

## 8. Categorias vs Variável Alvo (`delivery_late`)

Verificamos se existe relação entre variáveis categóricas e a probabilidade de atraso na entrega.

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(
    data=train_df,
    x="payment_type",
    hue="delivery_late",
    order=train_df["payment_type"].value_counts().index,
    palette="coolwarm"
)
plt.title("Tipo de Pagamento vs Atraso na Entrega")
plt.xlabel("Tipo de Pagamento")
plt.ylabel("Contagem")
plt.legend(title="Atrasado", labels=["Não", "Sim"])
plt.tight_layout()
plt.show()

In [ ]:
top_10_states = X_train["customer_state"].value_counts().head(10).index
mask = train_df["customer_state"].isin(top_10_states)

plt.figure(figsize=(13, 5))
sns.countplot(
    data=train_df[mask],
    x="customer_state",
    hue="delivery_late",
    order=top_10_states,
    palette="coolwarm"
)
plt.title("Estado do Cliente vs Atraso na Entrega (Top 10 estados)")
plt.xlabel("Estado")
plt.ylabel("Contagem")
plt.legend(title="Atrasado", labels=["Não", "Sim"])
plt.tight_layout()
plt.show()

## 9. Categorias vs Variáveis Numéricas (Boxplot)

Observamos como a distribuição de variáveis numéricas varia entre as categorias.

In [ ]:
# Limitamos ao percentil 95 para evitar distorção visual pelos outliers
cap = train_df["payment_value"].quantile(0.95)
plot_df = train_df[train_df["payment_value"] <= cap]

plt.figure(figsize=(10, 5))
sns.boxplot(
    data=plot_df,
    x="payment_type",
    y="payment_value",
    order=train_df["payment_type"].value_counts().index,
    palette="pastel"
)
plt.title("Valor do Pagamento por Tipo (sem outliers extremos)")
plt.xlabel("Tipo de Pagamento")
plt.ylabel("Valor do Pagamento (R$)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=train_df.dropna(subset=["review_score"]),
    x="delivery_late",
    y="review_score",
    palette="pastel"
)
plt.title("Nota da Avaliação vs Atraso na Entrega")
plt.xlabel("Atrasado (0 = Não, 1 = Sim)")
plt.ylabel("Nota da Avaliação")
plt.tight_layout()
plt.show()

## 10. Análise Temporal

Investigamos padrões de pedidos ao longo do tempo.

In [ ]:
train_orders = df.loc[X_train.index, ["order_purchase_timestamp"]].copy()
train_orders["month"]       = train_orders["order_purchase_timestamp"].dt.to_period("M")
train_orders["day_of_week"] = train_orders["order_purchase_timestamp"].dt.day_name()
train_orders["hour"]        = train_orders["order_purchase_timestamp"].dt.hour

monthly = train_orders["month"].value_counts().sort_index()

plt.figure(figsize=(14, 5))
monthly.plot(kind="bar", color="steelblue", edgecolor="white")
plt.title("Pedidos por Mês")
plt.xlabel("Mês")
plt.ylabel("Nº de Pedidos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_counts = train_orders["day_of_week"].value_counts().reindex(day_order)

plt.figure(figsize=(10, 5))
sns.barplot(x=dow_counts.index, y=dow_counts.values, palette="viridis")
plt.title("Pedidos por Dia da Semana")
plt.xlabel("Dia")
plt.ylabel("Nº de Pedidos")
plt.tight_layout()
plt.show()

In [ ]:
hour_counts = train_orders["hour"].value_counts().sort_index()

plt.figure(figsize=(12, 5))
sns.barplot(x=hour_counts.index, y=hour_counts.values, palette="viridis")
plt.title("Pedidos por Hora do Dia")
plt.xlabel("Hora")
plt.ylabel("Nº de Pedidos")
plt.tight_layout()
plt.show()

## 11. Correlações entre Variáveis Numéricas

Calculamos as correlações de Pearson entre as variáveis numéricas do conjunto de treino, incluindo o target `delivery_late`.

In [ ]:
corr_cols = ["total_price", "total_freight", "payment_value", "payment_installments", "n_items", "review_score", "delivery_late"]

corr_df = X_train[[c for c in corr_cols if c != "delivery_late"]].copy()
corr_df["delivery_late"] = y_train.values

corr_matrix = corr_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Matriz de Correlação (X_train)")
plt.tight_layout()
plt.show()

## Conclusões

Principais observações desta EDA:

- **Desbalanceamento do target:** a maioria dos pedidos é entregue no prazo; `delivery_late` é uma classe minoritária — isso deve ser considerado na modelagem.
- **Valores ausentes:** `review_score`, `payment_type` e algumas colunas de itens têm NaNs que precisarão de estratégia de imputação (mediana para numérica, moda para categórica).
- **Pagamentos:** cartão de crédito domina. Pedidos via boleto e voucher tendem a ter valores e perfis distintos.
- **Distribuição geográfica:** SP concentra a maior parte dos pedidos, mas estados mais distantes apresentam padrões diferentes de atraso.
- **Temporal:** há crescimento expressivo de pedidos ao longo de 2017-2018, com pico em novembro (Black Friday). Pedidos se concentram durante a semana e no horário comercial.
- **Avaliações:** pedidos com atraso recebem notas significativamente menores, confirmando impacto direto na experiência do cliente.
- **Correlações:** `total_price` e `payment_value` são altamente correlacionados (esperado). `delivery_late` apresenta correlação negativa com `review_score`.